# 🏙️ CSRNet: Crowd Density Estimation (Google Colab)

A **research-grade** crowd counting model using **CSRNet** (Congested Scene Recognition Network).

**Architecture:** VGG-16 frontend (pretrained on ImageNet) + dilated convolution backend

**Dataset:** ShanghaiTech Part B

**Paper:** [CSRNet: Dilated Convolutional Neural Networks for Understanding the Highly Congested Scenes (CVPR 2018)](https://arxiv.org/abs/1802.10062)

**Key Improvements over MC-CNN:**
- Pretrained VGG-16 feature extractor
- Dilated convolutions preserve spatial resolution
- Significantly better MAE/RMSE
- Exportable to TorchScript / ONNX for edge deployment

## 1. Setup & Environment

In [ ]:
# Install dependencies (Colab already has PyTorch, torchvision, etc.)
%pip install -q scipy matplotlib opencv-python-headless

import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import glob

import cv2
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import models

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Download ShanghaiTech Dataset

In [ ]:
# Option 1: Download via Kaggle API (recommended)
# Upload your kaggle.json to Colab first, or set env vars.

import os

DATASET_DIR = '/content/shanghaitech'

if not os.path.exists(DATASET_DIR):
    # Try Kaggle API
    try:
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

        # If kaggle.json is uploaded to Colab
        if os.path.exists('/content/kaggle.json'):
            !cp /content/kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json

        %pip install -q kaggle
        !kaggle datasets download -d tthien/shanghaitech -p /content/ --unzip
        print('Dataset downloaded via Kaggle API.')

    except Exception as e:
        print(f'Kaggle API failed: {e}')
        print('\n--- MANUAL DOWNLOAD ---')
        print('1. Download ShanghaiTech dataset from Kaggle:')
        print('   https://www.kaggle.com/datasets/tthien/shanghaitech')
        print('2. Upload and extract to /content/shanghaitech/')
        print('   The folder should contain ShanghaiTech/part_B/')
else:
    print('Dataset already exists.')

# Locate dataset - handle different extraction structures
possible_roots = [
    '/content/ShanghaiTech/part_B',
    '/content/shanghaitech/ShanghaiTech/part_B',
    '/content/shanghaitech/part_B',
]

DATA_ROOT = None
for p in possible_roots:
    if os.path.exists(p):
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        'Could not find ShanghaiTech part_B. '
        'Please check the download and extraction paths.'
    )

TRAIN_DIR = os.path.join(DATA_ROOT, 'train_data')
TEST_DIR = os.path.join(DATA_ROOT, 'test_data')

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'TRAIN_DIR: {TRAIN_DIR} (exists: {os.path.exists(TRAIN_DIR)})')
print(f'TEST_DIR:  {TEST_DIR} (exists: {os.path.exists(TEST_DIR)})')

## 3. Data Exploration

In [ ]:
# Load and visualize a sample image + ground truth
sample_img_dir = os.path.join(TRAIN_DIR, 'images')
sample_gt_dir = os.path.join(TRAIN_DIR, 'ground-truth')

img_files = sorted([f for f in os.listdir(sample_img_dir) if f.endswith('.jpg')])
print(f'Number of training images: {len(img_files)}')

# Pick a sample
sample_file = img_files[36] if len(img_files) > 36 else img_files[0]
img_path = os.path.join(sample_img_dir, sample_file)
img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

# Load ground truth
gt_filename = f'GT_{os.path.splitext(sample_file)[0]}.mat'
gt_path = os.path.join(sample_gt_dir, gt_filename)
gt = loadmat(gt_path)['image_info'][0][0][0][0][0]

# Draw markers
img_marked = img.copy()
for x, y in gt:
    cv2.drawMarker(img_marked, (int(x), int(y)), (255, 0, 0),
                   markerType=cv2.MARKER_CROSS, thickness=2, markerSize=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img)
axes[0].set_title(f'Original Image ({sample_file})')
axes[0].axis('off')
axes[1].imshow(img_marked)
axes[1].set_title(f'Ground Truth: {gt.shape[0]} people')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Density Map Generation

In [ ]:
def gen_density_map_gaussian(image, coords, sigma=5):
    """Generate a density map using Gaussian filter.
    
    Args:
        image: Input image (H, W, C) or (H, W)
        coords: Nx2 array of (x, y) head coordinates
        sigma: Gaussian kernel standard deviation
    
    Returns:
        density_map: (H, W) float32 array where sum ≈ number of people
    """
    h, w = image.shape[:2]
    density = np.zeros((h, w), dtype=np.float32)
    
    for x, y in coords:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= yi < h and 0 <= xi < w:
            density[yi, xi] = 1.0
    
    density_map = gaussian_filter(density, sigma=sigma, truncate=5.0)
    return density_map


# Visualize density map for sample image
dmap = gen_density_map_gaussian(img, gt, sigma=5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img)
axes[0].set_title(f'Original (GT count: {gt.shape[0]})')
axes[0].axis('off')
axes[1].imshow(dmap, cmap='jet')
axes[1].set_title(f'Density Map (sum: {dmap.sum():.2f})')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 5. Dataset Class

In [ ]:
class CrowdDataset(Dataset):
    """ShanghaiTech crowd counting dataset."""
    
    def __init__(self, root_dir, gt_downsample=4, augment=False):
        self.root_dir = root_dir
        self.gt_downsample = gt_downsample
        self.augment = augment
        
        img_dir = os.path.join(root_dir, 'images')
        self.img_names = sorted([
            f for f in os.listdir(img_dir) if f.endswith('.jpg')
        ])
        
        # Precompute density maps
        print(f'Loading {len(self.img_names)} images from {root_dir}...')
        self.n_people = {}
        self.density_maps = {}
        
        for fname in self.img_names:
            img_path = os.path.join(img_dir, fname)
            gt_fname = f'GT_{os.path.splitext(fname)[0]}.mat'
            gt_path = os.path.join(root_dir, 'ground-truth', gt_fname)
            
            gt_coords = loadmat(gt_path)['image_info'][0][0][0][0][0]
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            
            self.density_maps[fname] = gen_density_map_gaussian(img, gt_coords, sigma=5)
            self.n_people[fname] = gt_coords.shape[0]
        
        print(f'  Loaded. Total people annotations: {sum(self.n_people.values())}')
    
    def __len__(self):
        return len(self.img_names)
    
    def __getitem__(self, idx):
        fname = self.img_names[idx]
        img_path = os.path.join(self.root_dir, 'images', fname)
        
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        dmap = self.density_maps[fname].copy()
        n_people = self.n_people[fname]
        
        # Handle grayscale images
        if len(img.shape) == 2:
            img = np.stack([img]*3, axis=-1)
        
        # Resize to be divisible by gt_downsample
        ds = self.gt_downsample
        h, w = img.shape[:2]
        new_h = (h // ds) * ds
        new_w = (w // ds) * ds
        img = cv2.resize(img, (new_w, new_h))
        dmap = cv2.resize(dmap, (new_w // ds, new_h // ds))
        dmap = dmap[np.newaxis, :, :] * ds * ds
        
        # Data augmentation
        if self.augment and random.random() > 0.5:
            img = np.fliplr(img).copy()
            dmap = np.flip(dmap, axis=2).copy()
        
        # To tensors
        img_tensor = torch.from_numpy(img.transpose(2, 0, 1).astype(np.float32) / 255.0)
        dmap_tensor = torch.from_numpy(dmap.astype(np.float32))
        
        return img_tensor, dmap_tensor, n_people

## 6. Create DataLoaders

In [ ]:
batch_size = 4

# Training dataset (with augmentation)
full_train_dataset = CrowdDataset(TRAIN_DIR, gt_downsample=4, augment=True)

# Split train into train + validation (80/20)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Test dataset (no augmentation)
test_dataset = CrowdDataset(TEST_DIR, gt_downsample=4, augment=False)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

## 7. Visualize Training Samples

In [ ]:
def plot_batch(images, dmaps, n_cols=4, cmap='jet'):
    """Visualize a batch of images and their density maps."""
    n = images.shape[0]
    fig, axes = plt.subplots(2, n, figsize=(4*n, 6))
    if n == 1:
        axes = axes.reshape(2, 1)
    
    for i in range(n):
        axes[0, i].imshow(images[i].permute(1, 2, 0).cpu())
        axes[0, i].axis('off')
        
        dm = dmaps[i].squeeze().cpu().numpy()
        axes[1, i].imshow(dm, cmap=cmap)
        axes[1, i].set_title(f'Count: {dm.sum():.1f}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()


# Get one batch
sample_imgs, sample_dmaps, sample_counts = next(iter(train_loader))
plot_batch(sample_imgs, sample_dmaps)
print('Ground truth counts:', [int(c) for c in sample_counts])

## 8. CSRNet Model

**CSRNet** uses the first 10 layers of VGG-16 (pretrained on ImageNet) as a feature extractor,
followed by a backend of **dilated convolutions** that preserve spatial resolution while
capturing multi-scale context.

This is a significant upgrade over MC-CNN:
- Pretrained features → faster convergence, better generalization
- Dilated convolutions → larger receptive field without downsampling
- State-of-the-art results on ShanghaiTech

In [ ]:
class CSRNet(nn.Module):
    """CSRNet: Congested Scene Recognition Network.
    
    Frontend: VGG-16 layers (first 10 blocks, pretrained)
    Backend: Dilated convolutions for density estimation
    
    Reference: https://arxiv.org/abs/1802.10062
    """
    
    def __init__(self, pretrained=True):
        super(CSRNet, self).__init__()
        
        # Frontend: First 10 layers of VGG-16
        vgg = models.vgg16(weights='IMAGENET1K_V1' if pretrained else None)
        features = list(vgg.features.children())
        # Use layers up to relu4_3 (first 23 layers, before pool4)
        self.frontend = nn.Sequential(*features[:23])
        
        # Backend: Dilated convolutions
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 256, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
        )
        
        # Output layer
        self.output_layer = nn.Conv2d(64, 1, 1)
        
        # Initialize backend weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.backend.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        nn.init.normal_(self.output_layer.weight, std=0.01)
        nn.init.constant_(self.output_layer.bias, 0)
    
    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x


# Instantiate and inspect
model = CSRNet(pretrained=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'\nModel architecture:')
print(model)

## 9. Loss Function

In [ ]:
class CombinedLoss(nn.Module):
    """Combined pixel-level density map loss + count-level MAE loss."""
    
    def __init__(self, weight_dmap=0.8, weight_count=0.2):
        super().__init__()
        self.weight_dmap = weight_dmap
        self.weight_count = weight_count
        self.mse_loss = nn.MSELoss()
        self.mae_loss = nn.L1Loss()
    
    def forward(self, pred_dmap, gt_dmap, gt_count):
        gt_count = gt_count.float()
        
        # Pixel-level MSE loss
        dmap_loss = self.mse_loss(pred_dmap, gt_dmap)
        
        # Count-level MAE loss
        pred_count = pred_dmap.sum(dim=(2, 3)).squeeze()
        count_loss = self.mae_loss(pred_count, gt_count)
        
        # Combined
        total = self.weight_dmap * dmap_loss + self.weight_count * count_loss
        
        return total, count_loss

## 10. Training Loop

In [ ]:
num_epochs = 100  # Adjust based on your needs; more epochs = better convergence

# Model, loss, optimizer, scheduler
model = CSRNet(pretrained=True).to(device)
criterion = CombinedLoss(weight_dmap=0.8, weight_count=0.2)
optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# Tracking
train_losses, val_losses = [], []
train_maes, val_maes = [], []
best_val_mae = float('inf')
best_epoch = 0

SAVE_PATH = '/content/csrnet_crowd_counting_best.pth'

for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    tr_loss_sum, tr_mae_sum = 0.0, 0.0
    
    for imgs, dmaps, counts in train_loader:
        imgs, dmaps, counts = imgs.to(device), dmaps.to(device), counts.to(device).float()
        
        pred = model(imgs)
        loss, mae = criterion(pred, dmaps, counts)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        tr_loss_sum += loss.item() * imgs.size(0)
        tr_mae_sum += mae.item() * imgs.size(0)
    
    scheduler.step()
    tr_loss = tr_loss_sum / len(train_dataset)
    tr_mae = tr_mae_sum / len(train_dataset)
    
    # ---- Validation ----
    model.eval()
    val_loss_sum, val_mae_sum = 0.0, 0.0
    
    with torch.inference_mode():
        for imgs, dmaps, counts in val_loader:
            imgs, dmaps, counts = imgs.to(device), dmaps.to(device), counts.to(device).float()
            pred = model(imgs)
            loss, mae = criterion(pred, dmaps, counts)
            val_loss_sum += loss.item() * imgs.size(0)
            val_mae_sum += mae.item() * imgs.size(0)
    
    val_loss = val_loss_sum / len(val_dataset)
    val_mae = val_mae_sum / len(val_dataset)
    
    # Track
    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    train_maes.append(tr_mae)
    val_maes.append(val_mae)
    
    # Save best
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_epoch = epoch
        torch.save(model.state_dict(), SAVE_PATH)
    
    # Print every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:3d}/{num_epochs} | '
              f'Train Loss: {tr_loss:.6f} MAE: {tr_mae:.4f} | '
              f'Val Loss: {val_loss:.6f} MAE: {val_mae:.4f} | '
              f'LR: {lr:.2e}')

print(f'\n✅ Best epoch: {best_epoch+1} | Best Val MAE: {best_val_mae:.4f}')
print(f'Model saved to: {SAVE_PATH}')

## 11. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(train_losses, label='Train Loss', linewidth=2)
ax1.plot(val_losses, label='Val Loss', linewidth=2)
ax1.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best ({best_epoch+1})')
ax1.set_title('Training vs Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Combined Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# MAE
ax2.plot(train_maes, label='Train MAE', linewidth=2)
ax2.plot(val_maes, label='Val MAE', linewidth=2)
ax2.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best ({best_epoch+1})')
ax2.set_title('Training vs Validation MAE')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE (Count)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Evaluation on Test Set

In [ ]:
from math import sqrt

# Load best model
best_model = CSRNet(pretrained=False).to(device)
best_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
best_model.eval()

mae_total, mse_total = 0.0, 0.0

with torch.inference_mode():
    for imgs, dmaps, counts in test_loader:
        imgs = imgs.to(device)
        counts = counts.to(device).float()
        
        pred = best_model(imgs)
        pred_counts = pred.sum(dim=(2, 3)).squeeze()
        
        diff = pred_counts - counts
        mae_total += torch.abs(diff).sum().item()
        mse_total += (diff ** 2).sum().item()

n_test = len(test_dataset)
mae = mae_total / n_test
rmse = sqrt(mse_total / n_test)

print('=' * 40)
print('       TEST SET RESULTS')
print('=' * 40)
print(f'  MAE  = {mae:.3f}')
print(f'  RMSE = {rmse:.3f}')
print('=' * 40)

## 13. Visualize Predictions

In [ ]:
# Get a test batch
test_imgs, _, test_counts = next(iter(test_loader))
test_imgs = test_imgs.to(device)

with torch.inference_mode():
    pred_dmaps = best_model(test_imgs)
    pred_counts = pred_dmaps.sum(dim=(2, 3)).cpu().numpy().flatten()

# Visualize
n = min(4, test_imgs.shape[0])
fig, axes = plt.subplots(2, n, figsize=(5*n, 8))
if n == 1:
    axes = axes.reshape(2, 1)

for i in range(n):
    # Image
    axes[0, i].imshow(test_imgs[i].cpu().permute(1, 2, 0))
    axes[0, i].set_title(f'GT: {int(test_counts[i])}')
    axes[0, i].axis('off')
    
    # Predicted density map
    dm = pred_dmaps[i].squeeze().cpu().numpy()
    axes[1, i].imshow(dm, cmap='jet')
    axes[1, i].set_title(f'Pred: {pred_counts[i]:.1f}')
    axes[1, i].axis('off')

plt.suptitle('CSRNet Predictions on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Ground Truth: {[int(c) for c in test_counts[:n]]}')
print(f'Predicted:    {[round(float(c), 1) for c in pred_counts[:n]]}')

## 14. Model Export for Edge Deployment

Export the trained model in formats suitable for deployment:
- **TorchScript (.pt):** For PyTorch Mobile / LibTorch
- **ONNX (.onnx):** For TensorRT, OpenVINO, ONNX Runtime, etc.

In [ ]:
# === TorchScript Export ===
best_model.eval()
best_model_cpu = CSRNet(pretrained=False)
best_model_cpu.load_state_dict(torch.load(SAVE_PATH, map_location='cpu'))
best_model_cpu.eval()

# Trace with a dummy input
dummy_input = torch.randn(1, 3, 256, 256)
traced_model = torch.jit.trace(best_model_cpu, dummy_input)

TORCHSCRIPT_PATH = '/content/csrnet_crowd_counting.pt'
traced_model.save(TORCHSCRIPT_PATH)
print(f'✅ TorchScript model saved to: {TORCHSCRIPT_PATH}')
print(f'   File size: {os.path.getsize(TORCHSCRIPT_PATH) / 1e6:.1f} MB')

# === ONNX Export ===
ONNX_PATH = '/content/csrnet_crowd_counting.onnx'
torch.onnx.export(
    best_model_cpu,
    dummy_input,
    ONNX_PATH,
    input_names=['image'],
    output_names=['density_map'],
    dynamic_axes={
        'image': {0: 'batch', 2: 'height', 3: 'width'},
        'density_map': {0: 'batch', 2: 'height', 3: 'width'}
    },
    opset_version=11
)
print(f'✅ ONNX model saved to: {ONNX_PATH}')
print(f'   File size: {os.path.getsize(ONNX_PATH) / 1e6:.1f} MB')

# Verify TorchScript model
loaded = torch.jit.load(TORCHSCRIPT_PATH)
with torch.no_grad():
    out_original = best_model_cpu(dummy_input)
    out_traced = loaded(dummy_input)
    diff = (out_original - out_traced).abs().max().item()
    print(f'\n🔍 Verification: max difference = {diff:.2e} (should be ~0)')

## 15. Download Exported Models

In [ ]:
# Download the exported models from Colab
try:
    from google.colab import files
    
    print('📥 Downloading TorchScript model...')
    files.download(TORCHSCRIPT_PATH)
    
    print('📥 Downloading ONNX model...')
    files.download(ONNX_PATH)
    
    print('📥 Downloading best weights...')
    files.download(SAVE_PATH)
    
except ImportError:
    print('Not running in Colab. Models saved to:')
    print(f'  - {TORCHSCRIPT_PATH}')
    print(f'  - {ONNX_PATH}')
    print(f'  - {SAVE_PATH}')